Name: Christian Aron Vea

GitHub: https://github.com/ChristianAronVea/IND320-Vea

Streamlit: https://ind320-vea-5sc8vxsjg7v4bjgnpamvfl.streamlit.app/

Recording: https://nmbu.cloud.panopto.eu/Panopto/Pages/Viewer.aspx?id=7db6303f-2cb5-4f02-8134-b4fe010274d6


AI usage and log can be found in separate files in the repo.

In [10]:
import requests
import pandas as pd

from river import time_series, linear_model, preprocessing, optim, metrics

### Checking Norweigan bank API

In [16]:
# API for euro and usd from nowegian bank
url_bank = ("https://data.norges-bank.no/api/data/EXR/B.USD+EUR.NOK.SP?endPeriod=2026-11-26&format=sdmx-json&locale=no&startPeriod=2025-11-26")

r = requests.get(url_bank, timeout=10)
print(r.status_code)        # 200 is ok

data = r.json()
print(data.keys())

df_bank = pd.DataFrame(data)
df_bank

200
dict_keys(['meta', 'data'])


,meta,data
id,IREF066715,NaN
prepared,2026-11-26T15:24:35,NaN
test,False,NaN
datasetId,8f00749b-6950-4099-a89b-a13b8b4a08b9,NaN
sender,{'id': 'Unknown'},NaN
receiver,{'id': 'guest'},NaN
links,"[{'rel': 'self', 'href': '/data/EXR/B.USD+EUR....",NaN
dataSets,NaN,"[{'links': [{'rel': 'dataflow', 'urn': 'urn:sd..."
structure,NaN,"{'links': [{'rel': 'dataflow', 'urn': 'urn:sdm..."


In [21]:
print(df_bank["data"]["dataSets"])

[{'links': [{'rel': 'dataflow', 'urn': 'urn:sdmx:org.sdmx.infomodel.datastructure.Dataflow=NB:EXR(1.0)'}], 'reportingBegin': '2025-11-26T00:00:00', 'reportingEnd': '2026-11-26T23:59:59', 'action': 'Information', 'series': {'0:0:0:0': {'attributes': [0, 0, 0, 0], 'observations': {'0': ['10.233'], '1': ['10.1873'], '2': ['10.1716'], '3': ['10.1073'], '4': ['10.1507'], '5': ['10.0771'], '6': ['10.0994'], '7': ['10.0945'], '8': ['10.1124'], '9': ['10.1268'], '10': ['10.1504'], '11': ['10.0948'], '12': ['10.1066'], '13': ['10.1315'], '14': ['10.1775'], '15': ['10.225'], '16': ['10.201'], '17': ['10.1733'], '18': ['10.1213'], '19': ['10.0535'], '20': ['10.0518'], '21': ['10.0587'], '22': ['10.0791'], '23': ['10.0661'], '24': ['10.099'], '25': ['10.0184'], '26': ['10.0638'], '27': ['10.1015'], '28': ['10.1155'], '29': ['10.0449'], '30': ['10.0875'], '31': ['10.0704'], '32': ['10.0826'], '33': ['10.0889'], '34': ['10.0847'], '35': ['9.9893'], '36': ['9.9127'], '37': ['9.8889'], '38': ['9.8297'

In [ ]:
# Timezones
"""
From searching through previous assignments, I can find that all APIs use the same time zones - local or Europe/Oslo.
"""

### MakRiver Framework on SARIMAX

In [ ]:
magasinstatistikk = requests.get("https://biapi.nve.no/magasinstatistikk/api/Magasinstatistikk/HentOffentligData")

df_mag = pd.DataFrame(magasinstatistikk.json())
df_mag

,dato_Id,omrType,omrnr,iso_aar,iso_uke,fyllingsgrad,kapasitet_TWh,fylling_TWh,neste_Publiseringsdato,fyllingsgrad_forrige_uke,endring_fyllingsgrad
0,2020-11-15,EL,1,2020,46,0.974361,6.003264,5.849344,2020-11-25T13:00:00,0.997406,-0.023046
1,2011-08-28,EL,4,2011,34,0.786499,21.079208,16.578772,0001-01-01T00:00:00,0.787193,-0.000694
2,1998-07-05,EL,3,1998,27,0.793969,8.920932,7.082947,0001-01-01T00:00:00,0.733812,0.060157
3,2011-03-27,EL,4,2011,12,0.300820,21.079208,6.341054,0001-01-01T00:00:00,0.311122,-0.010301
4,2017-11-05,EL,4,2017,44,0.794520,21.079208,16.747847,0001-01-01T00:00:00,0.804085,-0.009565
...,...,...,...,...,...,...,...,...,...,...,...
14971,2026-11-22,VASS,1,2026,47,0.600108,36.044464,21.630564,2026-12-02T13:00:00,0.629544,-0.029436
14972,2026-11-22,VASS,2,2026,47,0.544164,23.237715,12.645121,2026-12-02T13:00:00,0.568755,-0.024591
14973,2026-11-15,VASS,1,2026,46,0.629544,36.044464,22.691559,2026-11-25T13:00:00,0.654478,-0.024935
14974,2026-11-22,VASS,3,2026,47,0.838650,28.155403,23.612534,2026-12-02T13:00:00,0.857217,-0.018566


In [ ]:

def fill_series(mag, omr_type, omr_nr):
    d = mag[(mag.omrType == omr_type) & (mag.omrnr == omr_nr)].copy()
    # ISO-år + uke -> mandag i den uken
    d["date"] = pd.to_datetime(d.iso_aar.astype(str) + d.iso_uke.astype(str).str.zfill(2) + "1",
                               format="%G%V%u")
    return d.set_index("date")["fyllingsgrad"].sort_index().asfreq("W-MON")

# Getting the currency rates fron NB
def get_nb_rates(start, end, currencies=("USD", "EUR")):
    """    Start and end has format:  yyyy-mm-dd """
    
    url = (f"https://data.norges-bank.no/api/data/EXR/B.{'+'.join(currencies)}.NOK.SP"
           f"?startPeriod={start}&endPeriod={end}&format=sdmx-json")
    j = requests.get(url, timeout=30).json()["data"]
    dates = [v["id"] for v in j["structure"]["dimensions"]["observation"][0]["values"]]
    base_dim = [d for d in j["structure"]["dimensions"]["series"] if d["id"] == "BASE_CUR"][0]
    rows = []
    for key, s in j["dataSets"][0]["series"].items():
        cur = base_dim["values"][int(key.split(":")[1])]["id"]
        for i, obs in s["observations"].items():
            rows.append((dates[int(i)], cur, float(obs[0])))
    df = pd.DataFrame(rows, columns=["date", "currency", "rate"])
    df["date"] = pd.to_datetime(df["date"])
    return df

In [27]:
bank_data = get_nb_rates("2026-02-01", "2026-02-03")
bank_data


,date,currency,rate
0,2026-02-02,USD,9.6837
1,2026-02-03,USD,9.6788
2,2026-02-02,EUR,11.4655
3,2026-02-03,EUR,11.4220


In [ ]:
import calendar, math, itertools
import pandas as pd
from river import time_series, compose, linear_model, preprocessing, optim, metrics

# --- Features (from the course book, 'month' -> 'date') ---
def get_month_distances(x):
    # RBF-like distance from the current month to each of the 12 months
    return {calendar.month_name[mo]: math.exp(-(x['date'].month - mo) ** 2)
            for mo in range(1, 13)}

def get_ordinal_date(x):
    # Day number since 0001-01-01, acts as a trend feature
    return {'ordinal_date': x['date'].toordinal()}

def build_model(exog_cols=(), p=1, d=0, q=0, m=52, sp=1, sd=0, sq=0,
                lr=0.01, intercept_init=0.0):
    cols = list(exog_cols)

    def get_exog(x):
        # Pick the selected exogenous variables from the input dict
        return {c: x[c] for c in cols}

    features = compose.TransformerUnion(get_ordinal_date, get_month_distances, get_exog)
    return features | time_series.SNARIMAX(
        p=p, d=d, q=q, m=m, sp=sp, sd=sd, sq=sq,
        regressor=preprocessing.StandardScaler() | linear_model.LinearRegression(
            intercept_init=intercept_init,   # as in the book: gives a better start
            optimizer=optim.SGD(lr),
            intercept_lr=0.3))

# --- Future exogenous values: mean per week number from the history ---
def climatology(d_, cols, fidx):
    if not cols:
        return pd.DataFrame(index=fidx)
    wk = d_[cols].groupby(d_.index.isocalendar().week).mean()
    # min(..., 52) guards against ISO week 53, which may be missing in the history
    return pd.DataFrame([wk.loc[min(ts.isocalendar().week, 52)] for ts in fidx], index=fidx)

# --- Main function ---
def forecast_fill(df, target='fylling', exog_cols=(), horizon=12, history=260,
                  x_future=None, **params):
    cols = list(exog_cols)
    # Keep only the last `history` rows and drop NaNs (River needs clean numbers)
    d_ = df[[target] + cols].dropna().iloc[-history:]
    model = build_model(cols, intercept_init=float(d_[target].iloc[0]), **params)

    # Train in chronological order, one observation at a time
    for ts, r in d_.iterrows():
        model.learn_one({'date': ts.date(), **{c: r[c] for c in cols}}, r[target])

    # Future weekly timestamps (Mondays), starting after the last training date
    fidx = pd.date_range(d_.index[-1], periods=horizon + 1, freq='W-MON')[1:]
    if x_future is None:
        x_future = climatology(d_, cols, fidx)
    # One dict per future step, as River's forecast() expects
    xs = [{'date': ts.date(), **{c: x_future.loc[ts, c] for c in cols}} for ts in fidx]
    return pd.Series(model.forecast(horizon=horizon, xs=xs), index=fidx, name='forecast')

# --- Hold-out test, same idea as in the book ---
def backtest(df, target='fylling', exog_cols=(), horizon=12, history=260, **params):
    cols = list(exog_cols)
    # Train on everything except the last `horizon` rows, compare against them
    train, test = df.iloc[:-horizon], df.iloc[-horizon:]
    fc = forecast_fill(train, target, cols, horizon, history,
                       x_future=test[cols],   # actual exogenous values in the test period
                       **params)
    mae, r2 = metrics.MAE(), metrics.R2()
    for yt, yp in zip(test[target], fc):
        mae.update(yt, yp)
        r2.update(yt, yp)
    return fc, mae.get(), r2.get()

NameError: name 'y' is not defined